# Making SIMSOPT GPU native: NVIDIA profile

Before running any cell, select **Runtime > Change runtime type > GPU**. This notebook refuses to profile if JAX falls back to the CPU. It clones the public feature branch, builds SIMSOPT, runs CPU/GPU parity checks, autotunes the Biot--Savart tile sizes, profiles the confirmed winner, and exports the sweep, a Perfetto trace, and a device-memory profile.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "gpu-native-objective",
            "https://github.com/PedroFranciscoGil/simsopt.git",
            str(repo),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True
    )
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True
)
os.chdir(repo)
source_root = repo / "src"
gpu_package = source_root / "simsopt" / "gpu" / "__init__.py"
assert gpu_package.is_file(), f"Missing GPU package at {gpu_package}"
source_path = str(source_root)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/gpu/test_biot_savart.py",
        "tests/gpu/test_objective.py",
        "tests/gpu/test_adapters.py",
        "tests/gpu/test_tile_sweep.py",
    ],
    check=True,
)

In [ ]:
import json
import shutil

artifact_root = Path("/content/simsopt-gpu-profile")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir(parents=True)
tile_sweep_file = artifact_root / "minimal-tile-sweep.json"
benchmark_env = os.environ.copy()
benchmark_env["OMP_NUM_THREADS"] = "1"
benchmark_env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

subprocess.run(
    [
        sys.executable,
        "benchmarks/gpu/sweep_gpu_tiles.py",
        "--problem",
        "minimal",
        "--warmup",
        "2",
        "--repeats",
        "5",
        "--top-k",
        "3",
        "--confirmation-warmup",
        "3",
        "--confirmation-repeats",
        "10",
        "--output",
        str(tile_sweep_file),
        "--quiet",
    ],
    check=True,
    env=benchmark_env,
)

In [ ]:
tile_sweep = json.loads(tile_sweep_file.read_text())
winner = tile_sweep["winner"]
print(json.dumps(winner, indent=2))

assert tile_sweep["environment"]["jax_backend"] == "gpu"
assert winner["confirmation_timing"]["median_seconds"] > 0
assert all(
    candidate["screening"]["eligible"]
    for candidate in tile_sweep["candidates"]
    if candidate["screening"]["status"] == "ok"
)

In [ ]:
result_file = artifact_root / "minimal-gpu.json"
trace_dir = artifact_root / "trace"
memory_file = artifact_root / "device-memory.prof"

subprocess.run(
    [
        sys.executable,
        "benchmarks/gpu/profile_gpu_objective.py",
        "--problem",
        "minimal",
        "--warmup",
        "3",
        "--repeats",
        "10",
        "--target-tile-size",
        str(winner["target_tile_size"]),
        "--source-tile-size",
        str(winner["source_tile_size"]),
        "--trace-dir",
        str(trace_dir),
        "--memory-profile",
        str(memory_file),
        "--output",
        str(result_file),
    ],
    check=True,
    env=benchmark_env,
)

In [ ]:
results = json.loads(result_file.read_text())
print(json.dumps(results, indent=2))

assert results["environment"]["jax_backend"] == "gpu"
assert results["parity"]["value_absolute_error"] < 1e-9
assert results["parity"]["curve_gradient_relative_l2_error"] < 1e-7

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-gpu-profile", "zip", artifact_root)
files.download(archive)